In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Using device: mps


In [7]:
import numpy as np

#Loading data
X_train = np.load("../data/processed/X_train.npy")
X_val = np.load("../data/processed/X_val.npy")
X_test = np.load("../data/processed/X_test.npy")
y_train = np.load("../data/processed/y_train.npy")
y_val = np.load("../data/processed/y_val.npy")
y_test = np.load("../data/processed/y_test.npy")

print(f"X_train shape:{X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")
print("Data loaded")

X_train shape:(686, 1997)
X_val shape: (172, 1997)
X_test shape: (215, 1997)
Data loaded


In [9]:
# Converting numpy arrays to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
X_test_t = torch.LongTensor(X_test).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
y_val_t = torch.LongTensor(y_val).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
print(f"Train batches: {len(train_loader)}")
print(f"val batches: {len(val_loader)}")
print(f"Input features: {X_train_t.shape[1]}")
print(f"Number of classes: {len(torch.unique(y_train_t))}")

Train batches: 22
val batches: 6
Input features: 1997
Number of classes: 4


In [10]:
class GeneExpressionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.3):
        super(GeneExpressionClassifier, self).__init__()

        self.network = nn.Sequential(
             #Layer 1
             nn.Linear(input_dim, hidden_dim),
             nn.BatchNorm1d(hidden_dim),
             nn.ReLU(),
             nn.Dropout(dropout),
             #Layer 2
             nn.Linear(hidden_dim, hidden_dim // 2),
             nn.BatchNorm1d(hidden_dim // 2),
             nn.ReLU(),
             nn.Dropout(dropout),
             #Layer 3
             nn.Linear(hidden_dim // 2, hidden_dim // 4),
             nn.BatchNorm1d(hidden_dim // 4),
             nn.ReLU(),
             nn.Dropout(dropout),
             #Output Layer
             nn.Linear(hidden_dim // 4, num_classes)
        )
    def forward(self, x):
        return self.network(x)

#Initalising model
model = GeneExpressionClassifier(
    input_dim = 1997, 
    hidden_dim = 512,
    num_classes = 4,
    dropout = 0.3
).to(device)

print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

GeneExpressionClassifier(
  (network): Sequential(
    (0): Linear(in_features=1997, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=4, bias=True)
  )
)
Total parameters: 1,189,508


In [13]:
class_counts = np.bincount(y_train)
class_weights = 1.0/class_counts
class_weights = class_weights/class_weights.sum()
class_weights_t = torch.FloatTensor(class_weights).to(device)

print("Class weights:")

class_names = ["NOS", "Ductal", "Lobular", "Mucinous"]
for name, weight, count in zip(class_names, class_weights, class_counts):
    print(f"{name}: weight={weight:.4f}, sample={count}")

#Function with class weights and optimiser
criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print("Training setup ready!!")

Class weights:
NOS: weight=0.1595, sample=48
Ductal: weight=0.0153, sample=499
Lobular: weight=0.0594, sample=129
Mucinous: weight=0.7658, sample=10
Training setup ready!!
